Surge-Sense: An ER Operations System built on RWFD of hospital patients' visits, the various metadata of those visits. The goal is to build a regressor to predict wait time. A sub-goal of this is building a forecaster to predict patient volume for a future day/time block; will be used as a feature in the regressor for predicting a patient's wait time. 

remember the context of the data; need to come back to this in eda/prelim analysis

### Clean & Split
Taking our dataset and creating two data products for the respective models.

In [ ]:
# dependecies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# google colab upload fix
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/er-data.csv')
data.head()

In [ ]:
# won't need patient name just id; verified proper name drop + also won't be using satisfaction score as a target
data = data.drop(['Patient First Inital', 'Patient Last Name', 'Patient Satisfaction Score'], axis=1)
data.head()

In [ ]:
# see inconsistencies with how gender is indentified
print(data['Patient Gender'].unique())

In [ ]:
# this fix works by finding the 'M' and 'F' in the gender column and replacing them with Male/Female
for x in data.index:
    if data.loc[x, 'Patient Gender'] == 'M':
        data.loc[x, 'Patient Gender'] = 'Male'
    if data.loc[x, 'Patient Gender'] == 'F':
        data.loc[x, 'Patient Gender'] = 'Female'
   
# verification      
print(data['Patient Gender'].unique())

In [ ]:
# need to change column names to fix the dates into days and times
data.rename({'Patient Id': 'patient_id', 
             'Patient Admission Date': 'admission_date',
             'Patient Gender': 'patient_gender',
             'Patient Age':  'patient_age',
             'Patient Race': 'patient_race',
             'Department Referral': 'department_referral',
             'Patient Admission Flag': 'admission_flag',
             'Patient Waittime': 'patient_waittime'}, axis=1, inplace=True)

In [ ]:
data.admission_date = data.admission_date.apply(pd.to_datetime)

In [ ]:
data['admission_day'] = [d.date() for d in data['admission_date']]
data['admission_time'] = [d.time() for d in data['admission_date']]

In [ ]:
data.head()

In [ ]:
data.drop(['admission_date'], axis=1, inplace=True)

data = data.loc[:, ['patient_id', 'patient_gender', 'patient_race', 'patient_age', 'admission_day', 'admission_time', 'patient_waittime', 'admission_flag', 'department_referral']]

In [ ]:
data.head()

In [ ]:
data['department_referral'] = data['department_referral'].fillna('No referral.')
data.head()

In [ ]:
data.describe()

In [ ]:
# feature engineering; basic binning
data['age_category'] = pd.cut(data['patient_age'], 
                              bins=[0, 19, 34, 44, 54, 64, 80], 
                              labels=['Minor', 'Young Adult', 'Adult', 'Middle-aged', 'Older Adult', 'Senior'])

In [ ]:
def time_category(hour):
    # using the wrap-around logic for midnight
    if 0 <= hour < 7:
        return 'Night'
    elif 7 <= hour < 15:
        return 'Day'
    elif 15 <= hour < 23:
        return 'Evening'
    return 'Night'

data['time_category'] = data['admission_time'].apply(
    lambda x: time_category(x.hour)
)

data.head()

In [ ]:
data['admission_day'] = pd.to_datetime(data['admission_day'])
data['day_of_week'] = data['admission_day'].dt.day_name()
data.head()